# Forgetting to Refuse — Phase 2 on Colab (A100)

Three sections, each ends with a push of its logs. Run them in order; a section can be resumed in a later session
because every output lives on Drive (`MyDrive/ftr/{data,runs,logs}`) and every step skips work that already exists.

**A. Data** (~3 h): render states, P baseline on the test states, scripted movement labels, RLDS exports, the A and C mixes.
**B. Align + gate** (~2 h): train A and C, score them offline, dev rollouts, Gate B numbers.
**C. Personalize + measure** (~10 h, can span sessions): N=50/200 personalization, hazard matrix, utility.

Every long cell streams one line per episode/update. `GH_TOKEN` secret required (same as Phase 1).

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, subprocess
os.environ.update(W='/content/ftr', DATA='/content/drive/MyDrive/ftr', REPO='/content/ftr/repo', HF='/content/ftr/hf', CUDA='cu118',
                  GH_TOKEN=userdata.get('GH_TOKEN'))
os.makedirs(os.environ['DATA'] + '/logs/phase2', exist_ok=True)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)

In [ ]:
%%shell
set -e
mkdir -p $W
[ -d $REPO ] || git clone -q https://$GH_TOKEN@github.com/ryan-racer/VLA_Alignment_Decay.git $REPO
git -C $REPO fetch -q && git -C $REPO reset -q --hard origin/main && git -C $REPO log --oneline | head -1
bash $REPO/scripts/setup_pod.sh 2>&1 | tail -2

## A. Data

In [ ]:
%%bash
set -o pipefail; pkill -f 'python -m ftr' 2>/dev/null; sleep 1
source $W/env.sh; L=$DATA/logs/phase2
[ -f $DATA/data/states_train.parquet ] || python -m ftr.build_data render --suite obstacle_avoidance_human --tasks 0 1 2 --states 0-29 --out $DATA/data/states_train.parquet 2>&1 | tee $L/a1_render_train.log | tail -1
[ -f $DATA/data/states_test.parquet ]  || python -m ftr.build_data render --suite obstacle_avoidance_human --tasks 3 4 --states 0-24 --out $DATA/data/states_test.parquet 2>&1 | tee $L/a2_render_test.log | tail -1
[ -f $DATA/data/pairs_test.parquet ]   || python -m ftr.build_data pairs --states $DATA/data/states_test.parquet --template-split test --out $DATA/data/pairs_test.parquet 2>&1 | tee $L/a3_pairs.log | tail -1
[ -f $DATA/data/pairs_dev.parquet ]    || python -m ftr.build_data pairs --states $DATA/data/states_train.parquet --template-split test --out $DATA/data/pairs_dev.parquet 2>&1 | tail -1
ls -la $DATA/data

In [ ]:
%%shell
set -o pipefail; pkill -f 'python -m ftr' 2>/dev/null; sleep 1
# scripted movement labels on the training states (no model, ~20 min)
source $W/env.sh; L=$DATA/logs/phase2
[ -f $DATA/runs/scripted/DONE ] || python -m ftr.rollout --resume --scripted --suite obstacle_avoidance_human --tasks 0 1 2 --states 0-29 --video 2 --out $DATA/runs/scripted 2>&1 | tee $L/a4_scripted.log | grep --line-buffered -E '^\[|Traceback|Error|wrote'
python - <<'EOF'
import pandas as pd, os
e = pd.read_parquet(os.environ['DATA'] + '/runs/scripted/episodes.parquet'); print(e['outcome'].value_counts().to_dict(), 'violation-free:', int((~e['contact']).sum()), 'of', len(e), '| DONE' if os.path.exists(os.environ['DATA'] + '/runs/scripted/DONE') else '| NOT DONE (partial)')
EOF

In [ ]:
%%shell
set -o pipefail; pkill -f 'python -m ftr' 2>/dev/null; sleep 1
# P baseline on the test states: matrix templates + blank (~2 h), then offline scoring on all test pairs
source $W/env.sh; L=$DATA/logs/phase2
[ -f $DATA/runs/P_hazard/DONE ] || python -m ftr.rollout --resume --ckpt $FTR_P_DIR --suite obstacle_avoidance_human --tasks 3 4 --states 0-24 \
  --classes harmful benign blank --templates h5 b5 z0 --video 2 --out $DATA/runs/P_hazard 2>&1 | tee $L/a5_P_hazard.log | grep --line-buffered -E '^\[|Traceback|Error|wrote'
[ -f $DATA/runs/P_score/predictions.parquet ] || python -m ftr.score --ckpt $FTR_P_DIR --pairs $DATA/data/pairs_test.parquet --out $DATA/runs/P_score 2>&1 | tee $L/a6_P_score.log | tail -8

In [ ]:
%%bash
set -o pipefail; pkill -f 'python -m ftr' 2>/dev/null; sleep 1
# RLDS exports (TF): rehearsal + personalization subsets. Same seed => N=50 is a prefix of N=200 per task.
source $W/env.sh; L=$DATA/logs/phase2
[ -f $DATA/data/spatial_rehearsal.parquet ] || python -m ftr.build_data export-rlds --rlds $FTR_RLDS_DIR/libero_spatial_no_noops --episodes 30 --seed 0 --category rehearsal --out $DATA/data/spatial_rehearsal.parquet 2>&1 | tee $L/a7_export.log | tail -1
[ -f $DATA/data/object_N200_p0.parquet ]    || python -m ftr.build_data export-rlds --rlds $FTR_RLDS_DIR/libero_object_no_noops --per-task 20 --seed 0 --out $DATA/data/object_N200_p0.parquet 2>&1 | tee -a $L/a7_export.log | tail -1
[ -f $DATA/data/object_N50_p0.parquet ]     || python -m ftr.build_data export-rlds --rlds $FTR_RLDS_DIR/libero_object_no_noops --per-task 5 --seed 0 --out $DATA/data/object_N50_p0.parquet 2>&1 | tee -a $L/a7_export.log | tail -1
ls -la $DATA/data

In [ ]:
%%bash
set -o pipefail; pkill -f 'python -m ftr' 2>/dev/null; sleep 1
# the arms
source $W/env.sh; L=$DATA/logs/phase2
for ARM in A C; do
  [ -f $DATA/data/$ARM.parquet ] || python -m ftr.build_data mix --arm $ARM --states $DATA/data/states_train.parquet --self $DATA/runs/scripted \
    --self-episodes $DATA/runs/scripted/episodes.parquet --rehearsal $DATA/data/spatial_rehearsal.parquet --out $DATA/data/$ARM.parquet 2>&1 | tee -a $L/a8_mix.log | tail -1
done

In [ ]:
%%bash
cd $REPO && mkdir -p logs/phase2 && cp $DATA/logs/phase2/*.log logs/phase2/ 2>/dev/null
git config user.email "rq11@rice.edu" && git config user.name "Ryan Quinlivan"
git add logs/phase2 && (git commit -qm "Phase 2 logs ($(date +%F_%H%M))" || echo "nothing to commit")
git fetch -q && git rebase -q origin/main && git push -q https://$GH_TOKEN@github.com/ryan-racer/VLA_Alignment_Decay.git HEAD:main && echo pushed

## B. Align + gate

In [ ]:
%%shell
set -o pipefail; pkill -f 'python -m ftr' 2>/dev/null; sleep 1
source $W/env.sh; L=$DATA/logs/phase2; cd $REPO
for ARM in A C; do
  [ -f $DATA/runs/${ARM}_s0/DONE ] || python -m ftr.finetune --vla_path $FTR_P_DIR --data_parquet $DATA/data/$ARM.parquet \
    --run_root_dir $DATA/runs --run_id ${ARM}_s0 --adapter_tmp_dir $DATA/runs/adapters \
    --epochs 3 --batch_size 8 --grad_accumulation_steps 2 --learning_rate 5e-4 --lora_rank 32 --seed 0 \
    2>&1 | tee $L/b1_train_$ARM.log | grep --line-buffered -E 'rows|updates|Saving|done|Traceback|Error|trainable'
done

In [ ]:
%%shell
set -o pipefail; pkill -f 'python -m ftr' 2>/dev/null; sleep 1
# reload check, then offline Ru/Rs/blank on the test pairs for A@0 and C@0
source $W/env.sh; L=$DATA/logs/phase2
FTR_RUN=$DATA/runs/A_s0 FTR_ADAPTER=$DATA/runs/adapters/A_s0 FTR_PAIRS=$DATA/data/pairs_test.parquet pytest tests/test_reload.py -m gpu -v 2>&1 | tee $L/b2_reload.log | tail -5
for ARM in A C; do
  [ -f $DATA/runs/${ARM}_s0/score_test/predictions.parquet ] || python -m ftr.score --ckpt $DATA/runs/${ARM}_s0 --pairs $DATA/data/pairs_test.parquet --out $DATA/runs/${ARM}_s0/score_test 2>&1 | tee $L/b3_score_$ARM.log | tail -6
done

In [ ]:
%%shell
set -o pipefail; pkill -f 'python -m ftr' 2>/dev/null; sleep 1
# Gate B dev rollouts: A@0 vs C@0 on 20 held-back training states, harmful + benign (~1 h)
source $W/env.sh; L=$DATA/logs/phase2
for ARM in A C; do
  [ -f $DATA/runs/${ARM}_s0/dev/DONE ] || python -m ftr.rollout --resume --ckpt $DATA/runs/${ARM}_s0 --suite obstacle_avoidance_human --tasks 0 1 2 --states 30-36 \
    --classes harmful benign --templates h5 b5 --out $DATA/runs/${ARM}_s0/dev 2>&1 | tee $L/b4_dev_$ARM.log | grep --line-buffered -E '^\[|Traceback|Error|wrote'
done
python -m ftr.analyze --runs $DATA/runs/P_score $DATA/runs/A_s0/score_test $DATA/runs/C_s0/score_test $DATA/runs/A_s0/dev $DATA/runs/C_s0/dev \
  --p $FTR_P_DIR --pairs $DATA/runs/A_s0:$DATA/runs/C_s0 --out $DATA/figures_gate 2>&1 | tee $L/b5_gate.log | head -60
cat $DATA/figures_gate/refusal_rates.csv; cat $DATA/figures_gate/contact_rates.csv

In [ ]:
%%shell
set -o pipefail; pkill -f 'python -m ftr' 2>/dev/null; sleep 1
# baseline half of the hazard matrix now (A@0, C@0), two processes; P's Uold. Survives a personalization failure.
source $W/env.sh; L=$DATA/logs/phase2
run() { [ -f $DATA/runs/$1/hazard/DONE ] || python -m ftr.rollout --resume --ckpt $DATA/runs/$1 --suite obstacle_avoidance_human --tasks 3 4 --states 0-24 \
    --classes harmful benign --templates h5 b5 --video 2 --out $DATA/runs/$1/hazard > $L/b6_hazard_$1.log 2>&1; echo "$1 exit $?"; }
run A_s0 & run C_s0 & wait
export LIBERO_DIR=$W/LIBERO LIBERO_CONFIG_PATH=$W/LIBERO/.libero_config PYTHONPATH=$REPO:$W/openvla:$W/LIBERO
[ -f $DATA/runs/P_u_libero_spatial/DONE ] || python -m ftr.rollout --resume --ckpt $FTR_P_DIR --suite libero_spatial --tasks all --states 0-4 --task-instruction \
    --out $DATA/runs/P_u_libero_spatial 2>&1 | tee $L/b7_P_uold.log | grep --line-buffered -E '^\[|timing|Traceback|Error|wrote'
grep -h -c '^\[' $L/b6_hazard_*.log

In [ ]:
%%bash
cd $REPO && mkdir -p logs/phase2 && cp $DATA/logs/phase2/*.log logs/phase2/ 2>/dev/null
git config user.email "rq11@rice.edu" && git config user.name "Ryan Quinlivan"
git add logs/phase2 && (git commit -qm "Phase 2 logs ($(date +%F_%H%M))" || echo "nothing to commit")
git fetch -q && git rebase -q origin/main && git push -q https://$GH_TOKEN@github.com/ryan-racer/VLA_Alignment_Decay.git HEAD:main && echo pushed

## C. Personalize + measure  (stop here until Gate B is read)

In [ ]:
%%shell
set -o pipefail; pkill -f 'python -m ftr' 2>/dev/null; sleep 1
source $W/env.sh; L=$DATA/logs/phase2; cd $REPO
for JOB in "A_s0 200" "C_s0 200" "A_s0 50"; do set -- $JOB; PARENT=$1; N=$2
  [ -f $DATA/runs/${PARENT}_N${N}_p0/DONE ] || python -m ftr.finetune --vla_path $DATA/runs/$PARENT --data_parquet $DATA/data/object_N${N}_p0.parquet \
    --run_root_dir $DATA/runs --run_id ${PARENT}_N${N}_p0 --adapter_tmp_dir $DATA/runs/adapters \
    --epochs 3 --batch_size 8 --grad_accumulation_steps 2 --learning_rate 5e-4 --lora_rank 32 --seed 0 \
    2>&1 | tee $L/c1_personalize_${PARENT}_N${N}.log | grep --line-buffered -E 'rows|updates|Saving|done|Traceback|Error'
done

In [ ]:
%%shell
set -o pipefail; pkill -f 'python -m ftr' 2>/dev/null; sleep 1
# offline Ru/Rs/blank at every endpoint
source $W/env.sh; L=$DATA/logs/phase2
for CK in A_s0_N50_p0 A_s0_N200_p0 C_s0_N200_p0; do
  [ -f $DATA/runs/$CK/score_test/predictions.parquet ] || python -m ftr.score --ckpt $DATA/runs/$CK --pairs $DATA/data/pairs_test.parquet --out $DATA/runs/$CK/score_test 2>&1 | tee $L/c2_score_$CK.log | tail -6
done

In [ ]:
%%shell
set -o pipefail; pkill -f 'python -m ftr' 2>/dev/null; sleep 1
# hazard matrix, personalized arms (A@0 / C@0 ran in section B). Two processes in parallel (~2 h).
source $W/env.sh; L=$DATA/logs/phase2
run() { [ -f $DATA/runs/$1/hazard/DONE ] || python -m ftr.rollout --resume --ckpt $DATA/runs/$1 --suite obstacle_avoidance_human --tasks 3 4 --states 0-24 \
    --classes harmful benign --templates h5 b5 --video 2 --out $DATA/runs/$1/hazard > $L/c3_hazard_$1.log 2>&1; echo "$1 exit $?"; }
run A_s0_N200_p0 & run C_s0_N200_p0 & wait
grep -h -c '^\[' $L/c3_hazard_*.log

In [ ]:
%%shell
set -o pipefail; pkill -f 'python -m ftr' 2>/dev/null; sleep 1
# utility: Uold (libero_spatial) and Unew (libero_object) at A@0 and A@200, 5 states/task (~3 h). Upstream LIBERO checkout.
source $W/env.sh; export LIBERO_DIR=$W/LIBERO LIBERO_CONFIG_PATH=$W/LIBERO/.libero_config PYTHONPATH=$REPO:$W/openvla:$W/LIBERO; L=$DATA/logs/phase2
for CK in A_s0 A_s0_N200_p0; do for SUITE in libero_spatial libero_object; do
  [ -f $DATA/runs/$CK/u_$SUITE/DONE ] || python -m ftr.rollout --resume --ckpt $DATA/runs/$CK --suite $SUITE --tasks all --states 0-4 --task-instruction \
    --out $DATA/runs/$CK/u_$SUITE 2>&1 | tee $L/c4_util_${CK}_$SUITE.log | grep --line-buffered -E '^\[|Traceback|Error|wrote'
done; done  # P's Uold ran in section B

In [ ]:
%%bash
set -o pipefail; pkill -f 'python -m ftr' 2>/dev/null; sleep 1
source $W/env.sh; L=$DATA/logs/phase2
python -m ftr.analyze --runs $DATA/runs/P_score $DATA/runs/*/score_test $DATA/runs/*/hazard $DATA/runs/*/u_* \
  --p $FTR_P_DIR --pairs $DATA/runs/A_s0:$DATA/runs/A_s0_N200_p0 $DATA/runs/C_s0:$DATA/runs/C_s0_N200_p0 --targets $DATA/data/A.parquet $DATA/data/C.parquet $DATA/data/object_N50_p0.parquet $DATA/data/object_N200_p0.parquet --out $DATA/figures 2>&1 | tee $L/c5_analyze.log | head -80
cp -r $DATA/figures $REPO/figures

In [ ]:
%%bash
cd $REPO && mkdir -p logs/phase2 && cp $DATA/logs/phase2/*.log logs/phase2/ 2>/dev/null
git config user.email "rq11@rice.edu" && git config user.name "Ryan Quinlivan"
git add logs/phase2 figures figures && (git commit -qm "Phase 2 logs ($(date +%F_%H%M))" || echo "nothing to commit")
git fetch -q && git rebase -q origin/main && git push -q https://$GH_TOKEN@github.com/ryan-racer/VLA_Alignment_Decay.git HEAD:main && echo pushed